In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score

import random
import os



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =========================
# LOAD N-BACK
# =========================
data_nb = torch.load(
    "/content/drive/MyDrive/eeg_preprocessed_nback.pt",
    weights_only=False
)

X_time_t = data_nb["X_time"]
X_bp_t   = data_nb["X_bp"]
X_cov_t  = data_nb["X_cov"]
X_plv_t  = data_nb["X_plv"]
y_t      = data_nb["y"]
subjects = data_nb["subjects"]

print("Loaded N-back")

Loaded N-back


In [ ]:
print("\n--- N-BACK ---")
print(X_time_t.shape, X_bp_t.shape, X_cov_t.shape, X_plv_t.shape, y_t.shape)


--- N-BACK ---
torch.Size([12159, 14, 512]) torch.Size([12159, 42]) torch.Size([15007, 105]) torch.Size([15007, 210]) torch.Size([12159])


In [ ]:
def extract_embeddings(model, loader, device):
    model.eval()
    E = []
    Y = []

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb, db in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            db      = db.to(device)

            # SAFE: supports both versions of model
            try:
                e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi, db)
            except TypeError:
                e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            E.append(e.cpu())
            Y.append(yb.cpu())

    return torch.cat(E, dim=0), torch.cat(Y, dim=0)

In [ ]:
import torch

def build_roi_workload_features_from_bp(
    X_bp_flat: torch.Tensor,          # (N, 42)
    n_ch: int = 14,
    bands=("theta", "alpha", "beta"),  # must match how you built X_bp
    frontal_idx=(0, 1, 2, 3),          # CHANGE THIS to match your montage
    parietal_idx=(10, 11, 12, 13),     # CHANGE THIS to match your montage
    use_log: bool = True,
    use_relative: bool = False,        # relative bandpower within channel (theta/alpha/beta)
    eps: float = 1e-8
) -> torch.Tensor:
    """
    Returns ROI features (N, 3):
      [frontal_theta, parietal_alpha, theta_alpha_ratio]
    """
    assert X_bp_flat.ndim == 2, "X_bp_flat must be (N, 42)"
    n_bands = len(bands)
    assert X_bp_flat.shape[1] == n_ch * n_bands, "bp dim mismatch"

    # reshape -> (N, ch, band)
    X = X_bp_flat.view(-1, n_ch, n_bands)

    # optional relative power within channel
    if use_relative:
        denom = X.sum(dim=2, keepdim=True).clamp_min(eps)
        X = X / denom

    # optional log (common for EEG power)
    if use_log:
        X = torch.log(X.clamp_min(eps))

    # band indices
    band_to_i = {b: i for i, b in enumerate(bands)}
    theta_i = band_to_i["theta"]
    alpha_i = band_to_i["alpha"]

    theta_ch = X[:, :, theta_i]  # (N, ch)
    alpha_ch = X[:, :, alpha_i]  # (N, ch)

    frontal_theta = theta_ch[:, list(frontal_idx)].mean(dim=1)   # (N,)
    parietal_alpha = alpha_ch[:, list(parietal_idx)].mean(dim=1) # (N,)

    # ratio: theta_frontal / alpha_parietal
    # If using log-space, ratio becomes (log theta - log alpha) which is great + stable.
    if use_log:
        theta_alpha_ratio = frontal_theta - parietal_alpha
    else:
        theta_alpha_ratio = frontal_theta / parietal_alpha.clamp_min(eps)

    roi = torch.stack([frontal_theta, parietal_alpha, theta_alpha_ratio], dim=1)  # (N,3)
    return roi

In [ ]:
def extract_embeddings(model, loader, device):

    model.eval()
    E = []
    Y = []

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            E.append(e.cpu())
            Y.append(yb.cpu())

    E = torch.cat(E, dim=0)
    Y = torch.cat(Y, dim=0)

    return E, Y

In [ ]:
import torch.nn as nn

def set_partial_bn_adapt(model: nn.Module, allow=("time_branch",)):
    """
    Put BN layers in *specified submodules* into train mode (update running stats).
    Keep all other BN in eval mode.
    Always keep Dropout in eval mode.
    """
    # Default everything to eval
    model.eval()

    # Dropout always eval
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.eval()

    # Enable BN train only in allowed named submodules
    for name, module in model.named_modules():
        if any(name.startswith(a) or (f".{a}." in name) or name.endswith(a) for a in allow):
            for m in module.modules():
                if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                    m.train()

In [ ]:
def prototype_loss(emb, y):
    """
    emb: (B, D) embedding vectors (from encoder)
    y:   (B,) class labels
    """

    B, D = emb.shape
    loss = 0.0
    count = 0

    classes = torch.unique(y)

    for c in classes:
        mask = (y == c)
        if mask.sum() < 2:
            continue

        class_emb = emb[mask]              # (Nc, D)
        proto = class_emb.mean(dim=0)      # (D,)

        loss += ((class_emb - proto)**2).sum(dim=1).mean()
        count += 1

    if count > 0:
        return loss / count
    else:
        return torch.tensor(0.0, device=emb.device)

In [ ]:
def supcon_diff_subject_loss(proj, y, subj, temperature=0.1):
    """
    proj: (B, D) projection vectors
    y:    (B,) labels
    subj: (B,) subject ids (can be ints)
    positives: same y AND different subj
    """
    proj = F.normalize(proj, dim=1)
    B = proj.size(0)

    sim = (proj @ proj.t()) / temperature            # (B,B)
    sim = sim - sim.max(dim=1, keepdim=True).values  # stabilize

    y = y.view(-1, 1)
    subj = subj.view(-1, 1)

    same_y = (y == y.t())
    diff_s = (subj != subj.t())
    pos_mask = same_y & diff_s

    # exclude self
    eye = torch.eye(B, device=proj.device, dtype=torch.bool)
    pos_mask = pos_mask & (~eye)
    neg_mask = ~eye

    exp_sim = torch.exp(sim) * neg_mask
    denom = exp_sim.sum(dim=1, keepdim=True) + 1e-12

    # If an anchor has 0 positives in batch, we skip it (loss=0 for that anchor)
    pos_counts = pos_mask.sum(dim=1)  # (B,)
    log_prob = sim - torch.log(denom)

    # mean log-prob over positives
    loss_per_i = torch.zeros(B, device=proj.device)
    valid = pos_counts > 0
    loss_per_i[valid] = -(log_prob * pos_mask.float()).sum(dim=1)[valid] / pos_counts[valid].float()

    return loss_per_i[valid].mean() if valid.any() else torch.tensor(0.0, device=proj.device)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FusionEncoder(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=42, roi_dim=3):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim   # 42 + 3 = 45

        # ------------------
        # TIME BRANCH
        # ------------------
        self.time_branch = nn.Sequential(
            nn.Conv1d(14, 14, kernel_size=7, padding=3, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 14, kernel_size=15, padding=7, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        # ------------------
        # BANDPOWER + ROI BRANCH
        # ------------------
        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),   # was 42 → now 45
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        # ------------------
        # COVARIANCE BRANCH
        # ------------------
        self.cov_branch = nn.Sequential(
            nn.Linear(105, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 32)
        )

        # ------------------
        # PLV BRANCH
        # ------------------
        self.plv_branch = nn.Sequential(
            nn.Linear(210, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 32)
        )

        # ------------------
        # FUSION
        # ------------------
        self.embed = nn.Sequential(
            nn.Linear(32 + 32 + 32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x_time, x_bp, x_cov, x_plv, x_roi):

        # ------------------
        # Time branch
        # ------------------
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)  # (B,32)

        # ------------------
        # Bandpower + ROI
        # ------------------
        bp_in = torch.cat([x_bp, x_roi], dim=1)  # (B,45)
        z_bp = self.bp_branch(bp_in)

        # ------------------
        # Covariance
        # ------------------
        z_cov = self.cov_branch(x_cov)

        # ------------------
        # PLV
        # ------------------
        z_plv = self.plv_branch(x_plv)

        # ------------------
        # Fusion
        # ------------------
        z = torch.cat([z_time, z_bp, z_cov, z_plv], dim=1)
        e = self.embed(z)

        return e

In [ ]:
import torch.nn as nn

class ContrastiveModel(nn.Module):
    def __init__(self,
                 emb_dim=64,
                 proj_dim=64,
                 bp_dim=42,
                 roi_dim=3):

        super().__init__()

        # Main encoder (learns embedding geometry)
        self.encoder = FusionEncoder(
            emb_dim=emb_dim,
            bp_dim=bp_dim,
            roi_dim=roi_dim
        )

        # Projection head for SupCon loss
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_cov, x_plv, x_roi):
        # Embedding
        e = self.encoder(x_time, x_bp, x_cov, x_plv, x_roi)

        # Projection (used only for contrastive loss)
        p = self.proj(e)

        return e, p

In [ ]:
import torch.nn.functional as F

def prototype_ce_loss(e, y, temperature=0.1):
    """
    e: (B, D) embeddings
    y: (B,) labels
    """

    classes = torch.unique(y)
    prototypes = []

    for c in classes:
        prototypes.append(e[y == c].mean(dim=0))

    prototypes = torch.stack(prototypes)  # (C, D)

    # normalize for cosine
    e_norm = F.normalize(e, dim=1)
    proto_norm = F.normalize(prototypes, dim=1)

    logits = e_norm @ proto_norm.T  # (B, C)
    logits = logits / temperature

    # map labels to [0, C-1]
    label_map = {c.item(): i for i, c in enumerate(classes)}
    y_mapped = torch.tensor([label_map[int(lbl.item())] for lbl in y],
                            device=y.device)

    loss = F.cross_entropy(logits, y_mapped)
    return loss

In [ ]:
all_subjects = np.unique(subjects)
print("\nSubjects list:", all_subjects)


Subjects list: ['subject_01' 'subject_02' 'subject_03' 'subject_04' 'subject_06'
 'subject_07' 'subject_08' 'subject_09' 'subject_10' 'subject_12'
 'subject_13' 'subject_14' 'subject_15']


In [ ]:
import numpy as np
from torch.utils.data import Dataset

class EEGDataset(Dataset):
    def __init__(self, X_time, X_bp, X_cov, X_plv, X_roi, y, subj_strs):
        self.X_time = X_time
        self.X_bp   = X_bp
        self.X_cov  = X_cov
        self.X_plv  = X_plv
        self.X_roi  = X_roi
        self.y      = y

        uniq = np.unique(subj_strs)
        self.subj_map = {s: i for i, s in enumerate(uniq)}
        self.subj = torch.tensor([self.subj_map[s] for s in subj_strs], dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_time[idx],
            self.X_bp[idx],
            self.X_cov[idx],
            self.X_plv[idx],
            self.X_roi[idx],
            self.y[idx],
            self.subj[idx]
        )

In [ ]:
from torch.utils.data import Sampler
import random
from collections import defaultdict

class SubjectBalancedBatchSampler(Sampler):
    """
    Creates batches containing samples from multiple subjects.
    Works with your existing EEGDataset (which has .subj tensor).
    """

    def __init__(self, dataset, batch_size, subjects_per_batch=4):
        self.dataset = dataset
        self.batch_size = batch_size
        self.subjects_per_batch = subjects_per_batch

        # Build index list per subject
        self.subj_to_indices = defaultdict(list)
        for idx, s in enumerate(dataset.subj.tolist()):
            self.subj_to_indices[s].append(idx)

        self.subjects = list(self.subj_to_indices.keys())

        # samples per subject inside batch
        self.samples_per_subject = batch_size // subjects_per_batch

    def __iter__(self):
        # Shuffle indices per subject
        for s in self.subjects:
            random.shuffle(self.subj_to_indices[s])

        # Create subject iterators
        subj_iters = {s: iter(self.subj_to_indices[s]) for s in self.subjects}

        while True:
            # randomly choose subjects for this batch
            chosen_subjects = random.sample(self.subjects, self.subjects_per_batch)

            batch = []

            for s in chosen_subjects:
                for _ in range(self.samples_per_subject):
                    try:
                        batch.append(next(subj_iters[s]))
                    except StopIteration:
                        return  # stop when any subject runs out

            if len(batch) == self.batch_size:
                yield batch
            else:
                return

    def __len__(self):
        # approximate number of batches
        min_samples = min(len(v) for v in self.subj_to_indices.values())
        return (min_samples // self.samples_per_subject)

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    # X_cov_train = X_cov_t[train_idx].clone()
    # X_cov_test  = X_cov_t[test_idx].clone()

    # cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    # cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    # X_cov_train = (X_cov_train - cov_mu) / cov_sd
    # X_cov_test  = (X_cov_test  - cov_mu) / cov_sd
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # =========================
    # ROI WORKLOAD FEATURES
    # # =========================
    # X_roi_train = build_roi_workload_features_from_bp(
    #     X_bp_train,
    #     # frontal_idx=(0,1,2,3),
    #     frontal_idx = (0,1,2,3),  # AF3, F7, F3, FC5
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )
    # X_roi_test = build_roi_workload_features_from_bp(
    #     X_bp_test,
    #     frontal_idx=(0,1,2,3),
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )

    # roi_mu = X_roi_train.mean(dim=0, keepdim=True)
    # roi_sd = X_roi_train.std(dim=0, keepdim=True) + 1e-6
    # X_roi_train = (X_roi_train - roi_mu) / roi_sd
    # X_roi_test  = (X_roi_test  - roi_mu) / roi_sd
    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL (metric version: NO n_classes arg)
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL (train prototypes -> test classify)
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            # cosine logits
            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            # map test labels to local indices
            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best encoder before adaptation
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # STEP 1: normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # =====================================
    # FEATURE RELIABILITY WEIGHTING
    # =====================================
    # compute variance of each embedding dimension
    var = e_support.var(dim=0)
    # reliability = inverse variance
    weights = 1.0 / (var + 1e-6)
    # normalize weights
    weights = weights / weights.mean()
    # apply weighting
    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    # normalize prototype vectors
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nMETRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6215 | best 0.6215
Epoch 02 | zero-shot acc 0.6540 | best 0.6540
Epoch 03 | zero-shot acc 0.6982 | best 0.6982
Epoch 04 | zero-shot acc 0.6761 | best 0.6982
Epoch 05 | zero-shot acc 0.6930 | best 0.6982
Epoch 06 | zero-shot acc 0.7119 | best 0.7119
Epoch 07 | zero-shot acc 0.7371 | best 0.7371
Epoch 08 | zero-shot acc 0.7802 | best 0.7802
Epoch 09 | zero-shot acc 0.7077 | best 0.7802
Epoch 10 | zero-shot acc 0.7014 | best 0.7802
AdaBN + Mahalanobis few-shot acc: 0.8058362007141113

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4558 | best 0.4558


In [ ]:
# =========================
# CHANNEL SUBSET
# =========================
SELECTED_CH_IDX = [0, 2, 3]  # AF3, F3, FC5

In [ ]:
teacher = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
opt = torch.optim.Adam(teacher.parameters(), lr=LR)

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

SELECTED_CH_IDX = [0, 2, 3]

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

overall_results = []

def compute_proto_logits(e, y):
    classes = torch.unique(y)
    protos = torch.stack([e[y == c].mean(dim=0) for c in classes])
    return F.normalize(e, dim=1) @ F.normalize(protos, dim=1).T

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # DATA PREP
    # =========================
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)
    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # 1. TRAIN TEACHER
    # =========================
    teacher = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt_teacher = torch.optim.Adam(teacher.parameters(), lr=LR)

    best_teacher_state = None
    best_acc = -1

    for ep in range(EPOCHS):
        teacher.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = teacher(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss = (
                prototype_ce_loss(e, yb)
                + LAMBDA * supcon_diff_subject_loss(proj, yb, sb)
                + PROTO_LAMBDA * prototype_loss(e, yb)
            )

            opt_teacher.zero_grad()
            loss.backward()
            opt_teacher.step()

        # quick zero-shot just for selection
        teacher.eval()
        with torch.no_grad():
            e_test, y_test_full = extract_embeddings(teacher, test_loader, device)
            acc = (e_test.norm(dim=1).mean()).item()  # dummy metric

        if acc > best_acc:
            best_acc = acc
            best_teacher_state = copy.deepcopy(teacher.state_dict())

    teacher.load_state_dict(best_teacher_state)
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad = False

    # =========================
    # 2. TRAIN STUDENT
    # =========================
    student = StudentModel(64, 64, 42).to(device)
    opt_student = torch.optim.Adam(student.parameters(), lr=LR)

    for ep in range(EPOCHS):
        student.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            with torch.no_grad():
                e_teacher, _ = teacher(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            xb_time_small = xb_time[:, SELECTED_CH_IDX, :]
            e_student, proj_student = student(xb_time_small, xb_bp)

            loss_base = (
                prototype_ce_loss(e_student, yb)
                + LAMBDA * supcon_diff_subject_loss(proj_student, yb, sb)
                + PROTO_LAMBDA * prototype_loss(e_student, yb)
            )

            logits_teacher = compute_proto_logits(e_teacher, yb)
            logits_student = compute_proto_logits(e_student, yb)

            loss_kd = F.kl_div(
                F.log_softmax(logits_student / 3.0, dim=1),
                F.softmax(logits_teacher / 3.0, dim=1),
                reduction="batchmean"
            ) * (3.0 ** 2)

            loss_feat = 1 - F.cosine_similarity(e_student, e_teacher, dim=1).mean()

            loss = loss_base + 0.5 * loss_kd + 0.25 * loss_feat

            opt_student.zero_grad()
            loss.backward()
            opt_student.step()

    # =========================
    # 3. FEW-SHOT (STUDENT)
    # =========================
    torch.manual_seed(1234 + fold_i)

    all_e, all_y = extract_embeddings_student(student, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        idx = (all_y == c).nonzero(as_tuple=True)[0]
        support_idx.append(idx[torch.randperm(len(idx))[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_loader = DataLoader(
        torch.utils.data.Subset(test_ds, support_idx.tolist()),
        batch_size=256
    )

    # 🔥 BN ADAPT (STUDENT FIXED VERSION)
    for p in student.parameters():
        p.requires_grad = False

    set_partial_bn_adapt(student, allow=("encoder.time_branch", "encoder.bp_branch"))

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, _, _ in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)

            xb_time_small = xb_time[:, SELECTED_CH_IDX, :]
            _ = student(xb_time_small, xb_bp)

    student.eval()

    # =========================
    # EMBEDDINGS AFTER ADAPT
    # =========================
    all_e, all_y = extract_embeddings_student(student, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # normalize
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # reliability weighting
    var = e_support.var(dim=0)
    weights = (1.0 / (var + 1e-6))
    weights = weights / weights.mean()

    e_support *= weights
    e_query   *= weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.T @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov = (1 - SHRINK)*cov + SHRINK*(avg_var*torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    acc = (pred == y_query).float().mean().item()

    print("STUDENT FEW-SHOT ACC:", acc)
    overall_results.append(acc)

print("FINAL MEAN:", np.mean(overall_results))


FOLD 1/13 | TEST SUBJECT = subject_01
STUDENT FEW-SHOT ACC: 0.782267153263092

FOLD 2/13 | TEST SUBJECT = subject_02
STUDENT FEW-SHOT ACC: 0.7189988493919373

FOLD 3/13 | TEST SUBJECT = subject_03
STUDENT FEW-SHOT ACC: 0.7709750533103943

FOLD 4/13 | TEST SUBJECT = subject_04
STUDENT FEW-SHOT ACC: 0.7502750158309937

FOLD 5/13 | TEST SUBJECT = subject_06
STUDENT FEW-SHOT ACC: 0.7733026742935181

FOLD 6/13 | TEST SUBJECT = subject_07
STUDENT FEW-SHOT ACC: 0.6469221711158752

FOLD 7/13 | TEST SUBJECT = subject_08
STUDENT FEW-SHOT ACC: 0.7218390703201294

FOLD 8/13 | TEST SUBJECT = subject_09
STUDENT FEW-SHOT ACC: 0.5538817644119263

FOLD 9/13 | TEST SUBJECT = subject_10
STUDENT FEW-SHOT ACC: 0.7117437720298767

FOLD 10/13 | TEST SUBJECT = subject_12
STUDENT FEW-SHOT ACC: 0.7567567825317383

FOLD 11/13 | TEST SUBJECT = subject_13
STUDENT FEW-SHOT ACC: 0.6613636016845703

FOLD 12/13 | TEST SUBJECT = subject_14
STUDENT FEW-SHOT ACC: 0.586980938911438

FOLD 13/13 | TEST SUBJECT = subject_15

In [ ]:
class StudentEncoder(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=42):
        super().__init__()

        self.time_branch = nn.Sequential(
            nn.Conv1d(3, 16, kernel_size=7, padding=3),
            nn.BatchNorm1d(16),
            nn.GELU(),

            nn.Conv1d(16, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.GELU(),

            nn.AdaptiveAvgPool1d(1)
        )

        self.bp_branch = nn.Sequential(
            nn.Linear(bp_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Linear(64, 32)
        )

        self.embed = nn.Sequential(
            nn.Linear(32 + 32, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Linear(64, emb_dim)
        )

    def forward(self, x_time, x_bp):
        z_time = self.time_branch(x_time).squeeze(-1)
        z_bp = self.bp_branch(x_bp)
        z = torch.cat([z_time, z_bp], dim=1)
        return self.embed(z)


class StudentModel(nn.Module):
    def __init__(self, emb_dim=64, proj_dim=64, bp_dim=42):
        super().__init__()

        self.encoder = StudentEncoder(emb_dim, bp_dim)

        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp):
        e = self.encoder(x_time, x_bp)
        p = self.proj(e)
        return e, p

In [ ]:
def extract_embeddings_student(model, loader, device):
    model.eval()
    E, Y = [], []

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)

            xb_time_small = xb_time[:, SELECTED_CH_IDX, :]

            e, _ = model(xb_time_small, xb_bp)
            E.append(e.cpu())
            Y.append(yb)

    return torch.cat(E), torch.cat(Y)

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

SELECTED_CH_IDX = [0, 2, 3]

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

def compute_proto_logits(e, y):
    classes = torch.unique(y)
    protos = torch.stack([e[y == c].mean(dim=0) for c in classes])
    return F.normalize(e, dim=1) @ F.normalize(protos, dim=1).T

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*120)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*120)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # DATA
    # =========================
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)
    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # 🔥 TEACHER TRAINING
    # =========================
    teacher = ContrastiveModel(64,64,42,0).to(device)
    opt_teacher = torch.optim.Adam(teacher.parameters(), lr=LR)

    best_state = None
    best_acc = -1

    print("\n🔥 TRAINING TEACHER")

    for ep in range(1, EPOCHS+1):
        teacher.train()

        tot_loss=tot_m=tot_c=tot_p=0
        n=0

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:

            xb_time, xb_bp, xb_cov, xb_plv, xb_roi = xb_time.to(device), xb_bp.to(device), xb_cov.to(device), xb_plv.to(device), xb_roi.to(device)
            yb, sb = yb.to(device), sb.to(device)

            e, proj = teacher(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_m = prototype_ce_loss(e, yb)
            loss_c = supcon_diff_subject_loss(proj, yb, sb)
            loss_p = prototype_loss(e, yb)

            loss = loss_m + LAMBDA*loss_c + PROTO_LAMBDA*loss_p

            opt_teacher.zero_grad()
            loss.backward()
            opt_teacher.step()

            tot_loss+=loss.item(); tot_m+=loss_m.item(); tot_c+=loss_c.item(); tot_p+=loss_p.item(); n+=1

        # ZERO SHOT
        teacher.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(teacher, train_eval_loader, device)
            e_test, y_test_full  = extract_embeddings(teacher, test_loader, device)

            protos = torch.stack([e_train[y_train_full==c].mean(0) for c in torch.unique(y_train_full)])
            logits = F.normalize(e_test,dim=1) @ F.normalize(protos,dim=1).T
            pred = logits.argmax(1)

            label_map = {int(c.item()):i for i,c in enumerate(torch.unique(y_train_full))}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred.device)

            acc = (pred==y_test_local).float().mean().item()

        # DEBUG STATS
        emb_norm = e_train.norm(dim=1).mean().item()
        proto_norm = protos.norm(dim=1).mean().item()

        if acc>best_acc:
            best_acc=acc
            best_state=copy.deepcopy(teacher.state_dict())

        print(f"[TEACHER][{ep}] loss={tot_loss/n:.4f} | metric={tot_m/n:.4f} | con={tot_c/n:.4f} | proto={tot_p/n:.4f}")
        print(f"             zero-shot={acc:.4f} | best={best_acc:.4f}")
        print(f"             emb_norm={emb_norm:.3f} | proto_norm={proto_norm:.3f}")

    teacher.load_state_dict(best_state)
    teacher.eval()

    # =========================
    # 🔥 TEACHER FEW-SHOT
    # =========================
    print("\n🧠 TEACHER FEW-SHOT")

    all_e, all_y = extract_embeddings(teacher, test_loader_full, device)

    support_idx=[]
    for c in torch.unique(all_y):
        idx=(all_y==c).nonzero(as_tuple=True)[0]
        support_idx.append(idx[torch.randperm(len(idx))[:FEW_SHOT_PER_CLASS]])
    support_idx=torch.cat(support_idx)

    mask=torch.zeros(len(all_y),dtype=torch.bool); mask[support_idx]=True

    e_support=F.normalize(all_e[mask].to(device),dim=1)
    e_query=F.normalize(all_e[~mask].to(device),dim=1)
    y_support=all_y[mask].to(device)
    y_query=all_y[~mask].to(device)

    var=e_support.var(0)
    w=(1/(var+1e-6)); w=w/w.mean()

    e_support*=w; e_query*=w

    protos=torch.stack([e_support[y_support==c].mean(0) for c in torch.unique(y_support)])
    protos=F.normalize(protos,dim=1)

    residuals=torch.cat([e_support[y_support==c]-protos[i] for i,c in enumerate(torch.unique(y_support))])

    D=residuals.shape[1]
    cov=(residuals.T@residuals)/(max(residuals.shape[0]-1,1))
    avg_var=torch.trace(cov)/D
    cov=(1-SHRINK)*cov+SHRINK*(avg_var*torch.eye(D,device=device))
    cov_inv=torch.linalg.pinv(cov)

    diffs=e_query.unsqueeze(1)-protos.unsqueeze(0)
    dists=torch.einsum("nkd,dd,nkd->nk",diffs,cov_inv,diffs)

    teacher_fs=(dists.argmin(1)==y_query).float().mean().item()
    print("🔥 TEACHER FEW-SHOT:",teacher_fs)

    # =========================
    # 🚀 STUDENT TRAIN
    # =========================
    student = StudentModel(64,64,42).to(device)
    opt = torch.optim.Adam(student.parameters(), lr=LR)

    print("\n🚀 TRAINING STUDENT")

    for ep in range(1,EPOCHS+1):
        student.train()

        tot=tot_kd=tot_feat=tot_base=0
        n=0

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:

            xb_time, xb_bp, xb_cov, xb_plv, xb_roi = xb_time.to(device), xb_bp.to(device), xb_cov.to(device), xb_plv.to(device), xb_roi.to(device)
            yb, sb = yb.to(device), sb.to(device)

            with torch.no_grad():
                e_t,_=teacher(xb_time,xb_bp,xb_cov,xb_plv,xb_roi)

            xb_small=xb_time[:,SELECTED_CH_IDX,:]
            e_s,proj_s=student(xb_small,xb_bp)

            base=prototype_ce_loss(e_s,yb)+LAMBDA*supcon_diff_subject_loss(proj_s,yb,sb)+PROTO_LAMBDA*prototype_loss(e_s,yb)

            logit_t=compute_proto_logits(e_t,yb)
            logit_s=compute_proto_logits(e_s,yb)

            kd=F.kl_div(F.log_softmax(logit_s/3,1),F.softmax(logit_t/3,1),reduction="batchmean")*(3**2)
            feat=1-F.cosine_similarity(e_s,e_t,dim=1).mean()

            loss=base+0.5*kd+0.25*feat

            opt.zero_grad(); loss.backward(); opt.step()

            tot+=loss.item(); tot_kd+=kd.item(); tot_feat+=feat.item(); tot_base+=base.item(); n+=1

        # STUDENT ZERO SHOT
        student.eval()
        with torch.no_grad():
            e_train,y_train_full=extract_embeddings_student(student,DataLoader(train_ds,256),device)
            e_test,y_test_full=extract_embeddings_student(student,test_loader,device)

            protos=torch.stack([e_train[y_train_full==c].mean(0) for c in torch.unique(y_train_full)])
            logits=F.normalize(e_test,dim=1)@F.normalize(protos,dim=1).T
            pred=logits.argmax(1)

            label_map={int(c.item()):i for i,c in enumerate(torch.unique(y_train_full))}
            y_test_local=torch.tensor([label_map[int(v.item())] for v in y_test_full],device=pred.device)

            acc=(pred==y_test_local).float().mean().item()

        sim=F.cosine_similarity(e_s,e_t,dim=1).mean().item()
        print(f"[STUDENT][{ep}] loss={tot/n:.4f} | base={tot_base/n:.4f} | kd={tot_kd/n:.4f} | feat={tot_feat/n:.4f}")
        print(f"               zero-shot={acc:.4f} | teacher-align={sim:.4f}")

    # =========================
    # 🎯 STUDENT FEW-SHOT
    # =========================
    print("\n🎯 STUDENT FEW-SHOT")

    all_e, all_y = extract_embeddings_student(student, test_loader_full, device)

    support_idx=[]
    for c in torch.unique(all_y):
        idx=(all_y==c).nonzero(as_tuple=True)[0]
        support_idx.append(idx[torch.randperm(len(idx))[:FEW_SHOT_PER_CLASS]])
    support_idx=torch.cat(support_idx)

    mask=torch.zeros(len(all_y),dtype=torch.bool); mask[support_idx]=True

    e_support=F.normalize(all_e[mask].to(device),dim=1)
    e_query=F.normalize(all_e[~mask].to(device),dim=1)
    y_support=all_y[mask].to(device)
    y_query=all_y[~mask].to(device)

    var=e_support.var(0)
    w=(1/(var+1e-6)); w=w/w.mean()

    e_support*=w; e_query*=w

    protos=torch.stack([e_support[y_support==c].mean(0) for c in torch.unique(y_support)])
    protos=F.normalize(protos,dim=1)

    residuals=torch.cat([e_support[y_support==c]-protos[i] for i,c in enumerate(torch.unique(y_support))])

    D=residuals.shape[1]
    cov=(residuals.T@residuals)/(max(residuals.shape[0]-1,1))
    avg_var=torch.trace(cov)/D
    cov=(1-SHRINK)*cov+SHRINK*(avg_var*torch.eye(D,device=device))
    cov_inv=torch.linalg.pinv(cov)

    diffs=e_query.unsqueeze(1)-protos.unsqueeze(0)
    dists=torch.einsum("nkd,dd,nkd->nk",diffs,cov_inv,diffs)

    student_fs=(dists.argmin(1)==y_query).float().mean().item()

    print("🔥 STUDENT FEW-SHOT:",student_fs)

    overall_results.append((teacher_fs, student_fs))

print("\nFINAL RESULTS")
for i,(t,s) in enumerate(overall_results):
    print(f"Fold {i}: Teacher {t:.4f} | Student {s:.4f}")

print("MEAN TEACHER:",sum(x[0] for x in overall_results)/len(overall_results))
print("MEAN STUDENT:",sum(x[1] for x in overall_results)/len(overall_results))


FOLD 1/13 | TEST SUBJECT = subject_01

🔥 TRAINING TEACHER
[TEACHER][1] loss=4.6773 | metric=0.8762 | con=6.0894 | proto=7.5640
             zero-shot=0.6067 | best=0.6067
             emb_norm=0.685 | proto_norm=0.516
[TEACHER][2] loss=3.9202 | metric=0.7392 | con=5.9314 | proto=2.1533
             zero-shot=0.6698 | best=0.6698
             emb_norm=0.774 | proto_norm=0.581
[TEACHER][3] loss=3.6967 | metric=0.6610 | con=5.8771 | proto=0.9719
             zero-shot=0.6993 | best=0.6993
             emb_norm=0.734 | proto_norm=0.596
[TEACHER][4] loss=3.5881 | metric=0.6021 | con=5.8249 | proto=0.7357
             zero-shot=0.6814 | best=0.6993
             emb_norm=0.736 | proto_norm=0.626
[TEACHER][5] loss=3.5156 | metric=0.5651 | con=5.7895 | proto=0.5574
             zero-shot=0.7024 | best=0.7024
             emb_norm=0.718 | proto_norm=0.612
[TEACHER][6] loss=3.4558 | metric=0.5245 | con=5.7579 | proto=0.5233
             zero-shot=0.7287 | best=0.7287
             emb_norm=0.733 

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

SELECTED_CH_IDX = [0, 2, 3]

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

overall_results = []

def compute_proto_logits(e, y):
    classes = torch.unique(y)
    protos = torch.stack([e[y == c].mean(dim=0) for c in classes])
    return F.normalize(e, dim=1) @ F.normalize(protos, dim=1).T

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # DATA PREP
    # =========================
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)
    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # STUDENT MODEL
    # =========================
    student = StudentModel(64, 64, 42).to(device)
    opt = torch.optim.Adam(student.parameters(), lr=LR)

    best_state = None
    best_acc = -1

    # =========================
    # TRAIN
    # =========================
    print("\n🚀 TRAINING STUDENT (NO TEACHER)")

    for ep in range(1, EPOCHS+1):
        student.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            xb_time_small = xb_time[:, SELECTED_CH_IDX, :]

            e, proj = student(xb_time_small, xb_bp)

            loss_metric = prototype_ce_loss(e, yb)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        student.eval()
        with torch.no_grad():

            # train prototypes
            e_train, y_train_full = extract_embeddings_student(student, train_loader, device)
            e_test,  y_test_full  = extract_embeddings_student(student, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred.device)

            acc = (pred == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(student.state_dict())

        print(f"[STUDENT][{ep}] loss={loss.item():.4f} | zero-shot={acc:.4f} | best={best_acc:.4f}")

    # load best
    student.load_state_dict(best_state)

    # =========================
    # FEW-SHOT (UNCHANGED)
    # =========================
    print("\n🎯 STUDENT FEW-SHOT")

    torch.manual_seed(1234 + fold_i)

    all_e, all_y = extract_embeddings_student(student, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        idx = (all_y == c).nonzero(as_tuple=True)[0]
        support_idx.append(idx[torch.randperm(len(idx))[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_loader = DataLoader(
        torch.utils.data.Subset(test_ds, support_idx.tolist()),
        batch_size=256
    )

    # BN adapt
    for p in student.parameters():
        p.requires_grad = False

    set_partial_bn_adapt(student, allow=("encoder.time_branch", "encoder.bp_branch"))

    with torch.no_grad():
        for xb_time, xb_bp, _, _, _, _, _ in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)

            xb_time_small = xb_time[:, SELECTED_CH_IDX, :]
            _ = student(xb_time_small, xb_bp)

    student.eval()

    # embeddings after adapt
    all_e, all_y = extract_embeddings_student(student, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # normalize
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # reliability weighting
    var = e_support.var(dim=0)
    weights = (1.0 / (var + 1e-6))
    weights = weights / weights.mean()

    e_support *= weights
    e_query   *= weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.T @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov = (1 - SHRINK)*cov + SHRINK*(avg_var*torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    acc = (pred == y_query).float().mean().item()

    print("🔥 STUDENT FEW-SHOT:", acc)

    overall_results.append(acc)

print("\nFINAL RESULTS")
print("Mean:", float(np.mean(overall_results)), "Std:", float(np.std(overall_results)))


FOLD 1/13 | TEST SUBJECT = subject_01

🚀 TRAINING STUDENT (NO TEACHER)
[STUDENT][1] loss=3.9406 | zero-shot=0.6299 | best=0.6299
[STUDENT][2] loss=3.7627 | zero-shot=0.6614 | best=0.6614
[STUDENT][3] loss=3.7558 | zero-shot=0.6414 | best=0.6614
[STUDENT][4] loss=3.6182 | zero-shot=0.6341 | best=0.6614
[STUDENT][5] loss=3.6442 | zero-shot=0.5825 | best=0.6614
[STUDENT][6] loss=3.5686 | zero-shot=0.6025 | best=0.6614
[STUDENT][7] loss=3.6260 | zero-shot=0.5889 | best=0.6614
[STUDENT][8] loss=3.5103 | zero-shot=0.5952 | best=0.6614
[STUDENT][9] loss=3.3415 | zero-shot=0.5836 | best=0.6614
[STUDENT][10] loss=3.3378 | zero-shot=0.6141 | best=0.6614

🎯 STUDENT FEW-SHOT
🔥 STUDENT FEW-SHOT: 0.7979798316955566

FOLD 2/13 | TEST SUBJECT = subject_02

🚀 TRAINING STUDENT (NO TEACHER)
[STUDENT][1] loss=3.8120 | zero-shot=0.4302 | best=0.4302
[STUDENT][2] loss=3.8186 | zero-shot=0.5453 | best=0.5453
[STUDENT][3] loss=3.5946 | zero-shot=0.4271 | best=0.5453
[STUDENT][4] loss=3.3765 | zero-shot=0.457

KeyboardInterrupt: 

In [ ]:
class StudentEncoder(nn.Module):
    def __init__(self, emb_dim=64):
        super().__init__()

        self.time_branch = nn.Sequential(
            nn.Conv1d(3, 16, kernel_size=7, padding=3),
            nn.BatchNorm1d(16),
            nn.GELU(),

            nn.Conv1d(16, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.GELU(),

            nn.AdaptiveAvgPool1d(1)
        )

        self.embed = nn.Sequential(
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Linear(64, emb_dim)
        )

    def forward(self, x_time):
        z_time = self.time_branch(x_time).squeeze(-1)
        return self.embed(z_time)

In [ ]:
class StudentModel(nn.Module):
    def __init__(self, emb_dim=64, proj_dim=64):
        super().__init__()

        self.encoder = StudentEncoder(emb_dim)

        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time):
        e = self.encoder(x_time)
        p = self.proj(e)
        return e, p

In [ ]:
def extract_embeddings_student(model, loader, device):
    model.eval()
    E, Y = [], []

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in loader:

            xb_time = xb_time.to(device)
            xb_time_small = xb_time[:, SELECTED_CH_IDX, :]

            e, _ = model(xb_time_small)

            E.append(e.cpu())
            Y.append(yb)

    return torch.cat(E), torch.cat(Y)

In [ ]:
BP_PER_CH = X_bp_t.shape[1] // 14  # should be 3

bp_idx = []
for ch in SELECTED_CH_IDX:
    start = ch * BP_PER_CH
    end   = start + BP_PER_CH
    bp_idx.extend(range(start, end))

print("Using BP indices:", bp_idx)

Using BP indices: [0, 1, 2, 6, 7, 8, 9, 10, 11]


In [ ]:
class StudentEncoder(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=9):  # 3 channels × 3 bands
        super().__init__()

        # TIME (3 channels)
        self.time_branch = nn.Sequential(
            nn.Conv1d(3, 16, kernel_size=7, padding=3),
            nn.BatchNorm1d(16),
            nn.GELU(),

            nn.Conv1d(16, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.GELU(),

            nn.AdaptiveAvgPool1d(1)
        )

        # BP (ONLY selected channels)
        self.bp_branch = nn.Sequential(
            nn.Linear(bp_dim, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Linear(32, 32)
        )

        self.embed = nn.Sequential(
            nn.Linear(32 + 32, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Linear(64, emb_dim)
        )

    def forward(self, x_time, x_bp):
        z_time = self.time_branch(x_time).squeeze(-1)
        z_bp   = self.bp_branch(x_bp)

        z = torch.cat([z_time, z_bp], dim=1)
        return self.embed(z)

In [ ]:
class StudentModel(nn.Module):
    def __init__(self, emb_dim=64, proj_dim=64, bp_dim=9):
        super().__init__()

        self.encoder = StudentEncoder(emb_dim, bp_dim)

        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp):
        e = self.encoder(x_time, x_bp)
        p = self.proj(e)
        return e, p

In [ ]:
def extract_embeddings_student(model, loader, device):
    model.eval()
    E, Y = [], []

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)

            xb_time_small = xb_time[:, SELECTED_CH_IDX, :]
            xb_bp_small   = xb_bp[:, bp_idx]

            e, _ = model(xb_time_small, xb_bp_small)

            E.append(e.cpu())
            Y.append(yb)

    return torch.cat(E), torch.cat(Y)

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# =========================
# CONFIG
# =========================
SELECTED_CH_IDX = [0, 2, 3]   # AF3, F3, FC5

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

overall_results = []

# =========================
# BUILD BP INDEX (CRITICAL)
# =========================
BP_PER_CH = X_bp_t.shape[1] // 14  # should be 3

bp_idx = []
for ch in SELECTED_CH_IDX:
    start = ch * BP_PER_CH
    end   = start + BP_PER_CH
    bp_idx.extend(range(start, end))

print("Using BP indices:", bp_idx)

# =========================
# FOLD LOOP
# =========================
for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # DATA PREP
    # =========================
    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6

    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6

    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # dummy placeholders (unused)
    zeros_train = torch.zeros((X_time_train.shape[0], 1))
    zeros_test  = torch.zeros((X_time_test.shape[0], 1))

    train_ds = EEGDataset(
        X_time_train, X_bp_train,
        zeros_train, zeros_train, zeros_train,
        y_train, subj_train
    )

    test_ds = EEGDataset(
        X_time_test, X_bp_test,
        zeros_test, zeros_test, zeros_test,
        y_test, subj_test
    )

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)
    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    student = StudentModel(emb_dim=64, proj_dim=64, bp_dim=len(bp_idx)).to(device)
    opt = torch.optim.Adam(student.parameters(), lr=LR)

    best_state = None
    best_acc = -1

    print("\n🚀 TRAINING STUDENT (3CH + BP)")

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        student.train()

        for xb_time, xb_bp, _, _, _, yb, sb in train_loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            # 🔥 CRITICAL: restrict inputs
            xb_time_small = xb_time[:, SELECTED_CH_IDX, :]
            xb_bp_small   = xb_bp[:, bp_idx]

            e, proj = student(xb_time_small, xb_bp_small)

            loss_metric = prototype_ce_loss(e, yb)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        student.eval()
        with torch.no_grad():

            e_train, y_train_full = extract_embeddings_student(student, train_loader, device)
            e_test,  y_test_full  = extract_embeddings_student(student, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred.device
            )

            acc = (pred == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(student.state_dict())

        print(f"[STUDENT][{ep}] loss={loss.item():.4f} | zero-shot={acc:.4f} | best={best_acc:.4f}")

    # load best
    student.load_state_dict(best_state)

    # =========================
    # FEW-SHOT
    # =========================
    print("\n🎯 STUDENT FEW-SHOT")

    torch.manual_seed(1234 + fold_i)

    all_e, all_y = extract_embeddings_student(student, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        idx = (all_y == c).nonzero(as_tuple=True)[0]
        support_idx.append(idx[torch.randperm(len(idx))[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_loader = DataLoader(
        torch.utils.data.Subset(test_ds, support_idx.tolist()),
        batch_size=256
    )

    # BN adapt
    for p in student.parameters():
        p.requires_grad = False

    set_partial_bn_adapt(student, allow=("encoder.time_branch", "encoder.bp_branch"))

    with torch.no_grad():
        for xb_time, xb_bp, _, _, _, _, _ in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)

            xb_time_small = xb_time[:, SELECTED_CH_IDX, :]
            xb_bp_small   = xb_bp[:, bp_idx]

            _ = student(xb_time_small, xb_bp_small)

    student.eval()

    # =========================
    # FINAL EMBEDDINGS
    # =========================
    all_e, all_y = extract_embeddings_student(student, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    var = e_support.var(dim=0)
    weights = (1.0 / (var + 1e-6))
    weights = weights / weights.mean()

    e_support *= weights
    e_query   *= weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat(
        [e_support[y_support == c] - protos[i] for i, c in enumerate(classes)],
        dim=0
    )

    D = residuals.shape[1]
    cov = (residuals.T @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov = (1 - SHRINK)*cov + SHRINK*(avg_var*torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    acc = (pred == y_query).float().mean().item()

    print("🔥 STUDENT FEW-SHOT:", acc)

    overall_results.append(acc)

# =========================
# FINAL
# =========================
print("\nFINAL RESULTS")
print("Mean:", float(np.mean(overall_results)), "Std:", float(np.std(overall_results)))

Using BP indices: [0, 1, 2, 6, 7, 8, 9, 10, 11]

FOLD 1/13 | TEST SUBJECT = subject_01

🚀 TRAINING STUDENT (3CH + BP)
[STUDENT][1] loss=3.9962 | zero-shot=0.4837 | best=0.4837
[STUDENT][2] loss=3.8204 | zero-shot=0.5300 | best=0.5300
[STUDENT][3] loss=3.8072 | zero-shot=0.5457 | best=0.5457
[STUDENT][4] loss=3.7749 | zero-shot=0.5794 | best=0.5794
[STUDENT][5] loss=3.7610 | zero-shot=0.5836 | best=0.5836
[STUDENT][6] loss=3.7110 | zero-shot=0.5846 | best=0.5846
[STUDENT][7] loss=3.7471 | zero-shot=0.5499 | best=0.5846
[STUDENT][8] loss=3.7315 | zero-shot=0.5678 | best=0.5846
[STUDENT][9] loss=3.7194 | zero-shot=0.6193 | best=0.6193
[STUDENT][10] loss=3.7614 | zero-shot=0.5773 | best=0.6193

🎯 STUDENT FEW-SHOT
🔥 STUDENT FEW-SHOT: 0.8372615575790405

FOLD 2/13 | TEST SUBJECT = subject_02

🚀 TRAINING STUDENT (3CH + BP)
[STUDENT][1] loss=4.0017 | zero-shot=0.4750 | best=0.4750
[STUDENT][2] loss=3.7979 | zero-shot=0.4718 | best=0.4750
[STUDENT][3] loss=3.7725 | zero-shot=0.3674 | best=0.475

In [ ]:
SELECTED_CH_IDX = [6, 7, 4]

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# =========================
# CONFIG
# =========================
SELECTED_CH_IDX = [6, 7, 4]

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

overall_results = []

# =========================
# BUILD BP INDEX (CRITICAL)
# =========================
BP_PER_CH = X_bp_t.shape[1] // 14  # should be 3

bp_idx = []
for ch in SELECTED_CH_IDX:
    start = ch * BP_PER_CH
    end   = start + BP_PER_CH
    bp_idx.extend(range(start, end))

print("Using BP indices:", bp_idx)

# =========================
# FOLD LOOP
# =========================
for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # DATA PREP
    # =========================
    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6

    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6

    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # dummy placeholders (unused)
    zeros_train = torch.zeros((X_time_train.shape[0], 1))
    zeros_test  = torch.zeros((X_time_test.shape[0], 1))

    train_ds = EEGDataset(
        X_time_train, X_bp_train,
        zeros_train, zeros_train, zeros_train,
        y_train, subj_train
    )

    test_ds = EEGDataset(
        X_time_test, X_bp_test,
        zeros_test, zeros_test, zeros_test,
        y_test, subj_test
    )

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)
    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    student = StudentModel(emb_dim=64, proj_dim=64, bp_dim=len(bp_idx)).to(device)
    opt = torch.optim.Adam(student.parameters(), lr=LR)

    best_state = None
    best_acc = -1

    print("\n🚀 TRAINING STUDENT (3CH + BP)")

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        student.train()

        for xb_time, xb_bp, _, _, _, yb, sb in train_loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            # 🔥 CRITICAL: restrict inputs
            xb_time_small = xb_time[:, SELECTED_CH_IDX, :]
            xb_bp_small   = xb_bp[:, bp_idx]

            e, proj = student(xb_time_small, xb_bp_small)

            loss_metric = prototype_ce_loss(e, yb)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        student.eval()
        with torch.no_grad():

            e_train, y_train_full = extract_embeddings_student(student, train_loader, device)
            e_test,  y_test_full  = extract_embeddings_student(student, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred.device
            )

            acc = (pred == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(student.state_dict())

        print(f"[STUDENT][{ep}] loss={loss.item():.4f} | zero-shot={acc:.4f} | best={best_acc:.4f}")

    # load best
    student.load_state_dict(best_state)

    # =========================
    # FEW-SHOT
    # =========================
    print("\n🎯 STUDENT FEW-SHOT")

    torch.manual_seed(1234 + fold_i)

    all_e, all_y = extract_embeddings_student(student, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        idx = (all_y == c).nonzero(as_tuple=True)[0]
        support_idx.append(idx[torch.randperm(len(idx))[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_loader = DataLoader(
        torch.utils.data.Subset(test_ds, support_idx.tolist()),
        batch_size=256
    )

    # BN adapt
    for p in student.parameters():
        p.requires_grad = False

    set_partial_bn_adapt(student, allow=("encoder.time_branch", "encoder.bp_branch"))

    with torch.no_grad():
        for xb_time, xb_bp, _, _, _, _, _ in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)

            xb_time_small = xb_time[:, SELECTED_CH_IDX, :]
            xb_bp_small   = xb_bp[:, bp_idx]

            _ = student(xb_time_small, xb_bp_small)

    student.eval()

    # =========================
    # FINAL EMBEDDINGS
    # =========================
    all_e, all_y = extract_embeddings_student(student, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    var = e_support.var(dim=0)
    weights = (1.0 / (var + 1e-6))
    weights = weights / weights.mean()

    e_support *= weights
    e_query   *= weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat(
        [e_support[y_support == c] - protos[i] for i, c in enumerate(classes)],
        dim=0
    )

    D = residuals.shape[1]
    cov = (residuals.T @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov = (1 - SHRINK)*cov + SHRINK*(avg_var*torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    acc = (pred == y_query).float().mean().item()

    print("🔥 STUDENT FEW-SHOT:", acc)

    overall_results.append(acc)

# =========================
# FINAL
# =========================
print("\nFINAL RESULTS")
print("Mean:", float(np.mean(overall_results)), "Std:", float(np.std(overall_results)))

Using BP indices: [18, 19, 20, 21, 22, 23, 12, 13, 14]

FOLD 1/13 | TEST SUBJECT = subject_01

🚀 TRAINING STUDENT (3CH + BP)
[STUDENT][1] loss=4.0449 | zero-shot=0.4932 | best=0.4932
[STUDENT][2] loss=3.9545 | zero-shot=0.5237 | best=0.5237
[STUDENT][3] loss=3.9337 | zero-shot=0.6078 | best=0.6078
[STUDENT][4] loss=3.8622 | zero-shot=0.6278 | best=0.6278
[STUDENT][5] loss=3.8556 | zero-shot=0.5931 | best=0.6278
[STUDENT][6] loss=3.7797 | zero-shot=0.6267 | best=0.6278
[STUDENT][7] loss=3.8805 | zero-shot=0.6404 | best=0.6404
[STUDENT][8] loss=3.7478 | zero-shot=0.6698 | best=0.6698
[STUDENT][9] loss=3.7314 | zero-shot=0.6372 | best=0.6698
[STUDENT][10] loss=3.8389 | zero-shot=0.6456 | best=0.6698

🎯 STUDENT FEW-SHOT
🔥 STUDENT FEW-SHOT: 0.782267153263092

FOLD 2/13 | TEST SUBJECT = subject_02

🚀 TRAINING STUDENT (3CH + BP)
[STUDENT][1] loss=4.0342 | zero-shot=0.4601 | best=0.4601
[STUDENT][2] loss=3.9011 | zero-shot=0.5325 | best=0.5325
[STUDENT][3] loss=3.8591 | zero-shot=0.5314 | best

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# =========================
# CONFIG
# =========================
SELECTED_CH_IDX = [6, 7, 4]  # try "bad" electrodes

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

overall_results = []

# =========================
# SIMPLE BANDPOWER (LOCAL ONLY)
# =========================
def compute_bandpower(x):
    # x: [B, C, T]
    fft = torch.fft.rfft(x, dim=-1)
    psd = (fft.abs() ** 2)

    # crude bands (assumes fixed sampling)
    bands = [
        (1,4), (4,8), (8,13), (13,30)
    ]

    bp = []
    for low, high in bands:
        bp.append(psd[..., low:high].mean(dim=-1))

    return torch.cat(bp, dim=1)  # [B, C*bands]

# =========================
# FOLD LOOP
# =========================
for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # 🔥 CRITICAL: SELECT CHANNELS FIRST
    # =========================
    X_time_train = X_time_train[:, SELECTED_CH_IDX, :]
    X_time_test  = X_time_test[:, SELECTED_CH_IDX, :]

    # =========================
    # NORMALIZE (LOCAL ONLY)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6

    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    # =========================
    # DATASET (BP computed ON THE FLY)
    # =========================
    class LocalEEGDataset(torch.utils.data.Dataset):
        def __init__(self, X_time, y, subj):
            self.X_time = X_time
            self.y = y
            self.subj = subj

        def __len__(self):
            return len(self.y)

        def __getitem__(self, idx):
            return self.X_time[idx], self.y[idx], self.subj[idx]

    train_ds = LocalEEGDataset(X_time_train, y_train, subj_train)
    test_ds  = LocalEEGDataset(X_time_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # ✔ CORRECT eval loader
    train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    student = StudentModel(emb_dim=64, proj_dim=64, bp_dim=3*4).to(device)
    opt = torch.optim.Adam(student.parameters(), lr=LR)

    best_state = None
    best_acc = -1

    print("\n🚀 TRAINING STUDENT (TRUE 3CH)")

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        student.train()
        total_loss = 0

        for xb_time, yb, sb in train_loader:

            xb_time = xb_time.to(device)
            yb = yb.to(device)
            sb = sb.to(device)

            # 🔥 recompute BP ONLY from selected channels
            xb_bp = compute_bandpower(xb_time)

            e, proj = student(xb_time, xb_bp)

            loss_m = prototype_ce_loss(e, yb)
            loss_c = supcon_diff_subject_loss(proj, yb, sb)
            loss_p = prototype_loss(e, yb)

            loss = loss_m + LAMBDA*loss_c + PROTO_LAMBDA*loss_p

            opt.zero_grad()
            loss.backward()
            opt.step()

            total_loss += loss.item()

        # =========================
        # ZERO-SHOT
        # =========================
        student.eval()
        with torch.no_grad():

            def get_emb(loader):
                all_e, all_y = [], []
                for xb_time, yb, _ in loader:
                    xb_time = xb_time.to(device)
                    xb_bp = compute_bandpower(xb_time)
                    e, _ = student(xb_time, xb_bp)
                    all_e.append(e)
                    all_y.append(yb)
                return torch.cat(all_e), torch.cat(all_y)

            e_train, y_train_full = get_emb(train_eval_loader)
            e_test,  y_test_full  = get_emb(test_loader)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full==c].mean(0) for c in classes])

            logits = F.normalize(e_test,dim=1) @ F.normalize(protos,dim=1).T
            pred = logits.argmax(1)

            label_map = {int(c.item()):i for i,c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred.device
            )

            acc = (pred==y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(student.state_dict())

        print(f"[EP {ep}] loss={total_loss:.3f} | zero-shot={acc:.4f} | best={best_acc:.4f}")

    # =========================
    # LOAD BEST
    # =========================
    student.load_state_dict(best_state)

    # =========================
    # FEW-SHOT
    # =========================
    print("\n🎯 FEW-SHOT")

    student.eval()
    with torch.no_grad():
        e_all, y_all = get_emb(test_loader)

    support_idx = []
    for c in torch.unique(y_all):
        idx = (y_all == c).nonzero(as_tuple=True)[0]
        support_idx.append(idx[torch.randperm(len(idx))[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(y_all), dtype=torch.bool)
    mask[support_idx] = True

    e_support = F.normalize(e_all[mask].to(device), dim=1)
    e_query   = F.normalize(e_all[~mask].to(device), dim=1)

    y_support = y_all[mask].to(device)
    y_query   = y_all[~mask].to(device)

    protos = torch.stack([e_support[y_support==c].mean(0) for c in torch.unique(y_support)])
    protos = F.normalize(protos, dim=1)

    logits = e_query @ protos.T
    pred = logits.argmax(1)

    acc = (pred == y_query).float().mean().item()

    print("🔥 FEW-SHOT:", acc)

    overall_results.append(acc)

# =========================
# FINAL
# =========================
print("\nFINAL RESULTS")
print("Mean:", float(np.mean(overall_results)))
print("Std:", float(np.std(overall_results)))


FOLD 1/13 | TEST SUBJECT = subject_01

🚀 TRAINING STUDENT (TRUE 3CH)


AttributeError: 'tuple' object has no attribute 'to'

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# =========================
# CONFIG
# =========================
SELECTED_CH_IDX = [6, 7, 4]   # change electrodes here
# =========================
# CONVERT SUBJECTS TO INT IDS
# =========================
unique_subjects = np.unique(subjects)
subj_to_id = {s: i for i, s in enumerate(unique_subjects)}

subjects_num = np.array([subj_to_id[s] for s in subjects])

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

overall_results = []

# =========================
# BANDPOWER (LOCAL ONLY)
# =========================
def compute_bandpower(x):
    fft = torch.fft.rfft(x, dim=-1)
    psd = (fft.abs() ** 2)

    bands = [(1,4), (4,8), (8,13), (13,30)]

    bp = []
    for low, high in bands:
        bp.append(psd[..., low:high].mean(dim=-1))

    return torch.cat(bp, dim=1)  # [B, C*bands]

# =========================
# DATASET FIX (IMPORTANT)
# =========================
class LocalEEGDataset(torch.utils.data.Dataset):
    def __init__(self, X_time, y, subj):
        self.X_time = X_time
        self.y = y
        self.subj = subj

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_time[idx],
            self.y[idx],
            torch.tensor(self.subj[idx])  # 🔥 FIX HERE
        )

# =========================
# FOLD LOOP
# =========================
for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # SPLIT
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    # subj_train = subjects[train_idx]
    # subj_test  = subjects[test_idx]
    subj_train = subjects_num[train_idx]
    subj_test  = subjects_num[test_idx]

    # =========================
    # 🔥 SELECT CHANNELS FIRST
    # =========================
    X_time_train = X_time_train[:, SELECTED_CH_IDX, :]
    X_time_test  = X_time_test[:, SELECTED_CH_IDX, :]

    # =========================
    # NORMALIZATION
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6

    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    # DATASETS
    train_ds = LocalEEGDataset(X_time_train, y_train, subj_train)
    test_ds  = LocalEEGDataset(X_time_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)
    train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    student = StudentModel(emb_dim=64, proj_dim=64, bp_dim=3*4).to(device)
    opt = torch.optim.Adam(student.parameters(), lr=LR)

    best_state = None
    best_acc = -1

    print("\n🚀 TRAINING STUDENT (TRUE 3CH)")

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        student.train()
        total_loss = 0

        for xb_time, yb, sb in train_loader:

            xb_time = xb_time.to(device)
            yb = yb.to(device)
            sb = sb.to(device)  # ✅ NOW WORKS

            xb_bp = compute_bandpower(xb_time)

            e, proj = student(xb_time, xb_bp)

            loss_m = prototype_ce_loss(e, yb)
            loss_c = supcon_diff_subject_loss(proj, yb, sb)
            loss_p = prototype_loss(e, yb)

            loss = loss_m + LAMBDA*loss_c + PROTO_LAMBDA*loss_p

            opt.zero_grad()
            loss.backward()
            opt.step()

            total_loss += loss.item()

        # =========================
        # ZERO-SHOT
        # =========================
        student.eval()
        with torch.no_grad():

            def get_emb(loader):
                all_e, all_y = [], []
                for xb_time, yb, _ in loader:
                    xb_time = xb_time.to(device)
                    xb_bp = compute_bandpower(xb_time)
                    e, _ = student(xb_time, xb_bp)
                    all_e.append(e)
                    all_y.append(yb)
                return torch.cat(all_e), torch.cat(all_y)

            e_train, y_train_full = get_emb(train_eval_loader)
            e_test,  y_test_full  = get_emb(test_loader)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full==c].mean(0) for c in classes])

            logits = F.normalize(e_test,dim=1) @ F.normalize(protos,dim=1).T
            pred = logits.argmax(1)

            label_map = {int(c.item()):i for i,c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred.device
            )

            acc = (pred==y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(student.state_dict())

        print(f"[EP {ep}] loss={total_loss:.3f} | zero-shot={acc:.4f} | best={best_acc:.4f}")

    # =========================
    # LOAD BEST
    # =========================
    student.load_state_dict(best_state)

    # =========================
    # FEW-SHOT
    # =========================
    print("\n🎯 FEW-SHOT")

    student.eval()
    with torch.no_grad():
        e_all, y_all = get_emb(test_loader)

    support_idx = []
    for c in torch.unique(y_all):
        idx = (y_all == c).nonzero(as_tuple=True)[0]
        support_idx.append(idx[torch.randperm(len(idx))[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(y_all), dtype=torch.bool)
    mask[support_idx] = True

    e_support = F.normalize(e_all[mask].to(device), dim=1)
    e_query   = F.normalize(e_all[~mask].to(device), dim=1)

    y_support = y_all[mask].to(device)
    y_query   = y_all[~mask].to(device)

    protos = torch.stack([e_support[y_support==c].mean(0) for c in torch.unique(y_support)])
    protos = F.normalize(protos, dim=1)

    logits = e_query @ protos.T
    pred = logits.argmax(1)

    acc = (pred == y_query).float().mean().item()

    print("🔥 FEW-SHOT:", acc)

    overall_results.append(acc)

# =========================
# FINAL
# =========================
print("\nFINAL RESULTS")
print("Mean:", float(np.mean(overall_results)))
print("Std:", float(np.std(overall_results)))


FOLD 1/13 | TEST SUBJECT = subject_01

🚀 TRAINING STUDENT (TRUE 3CH)
[EP 1] loss=120.746 | zero-shot=0.3449 | best=0.3449
[EP 2] loss=111.075 | zero-shot=0.5163 | best=0.5163
[EP 3] loss=109.461 | zero-shot=0.6435 | best=0.6435
[EP 4] loss=108.753 | zero-shot=0.6740 | best=0.6740
[EP 5] loss=107.763 | zero-shot=0.6772 | best=0.6772
[EP 6] loss=107.121 | zero-shot=0.6730 | best=0.6772
[EP 7] loss=106.761 | zero-shot=0.6719 | best=0.6772
[EP 8] loss=106.089 | zero-shot=0.6688 | best=0.6772
[EP 9] loss=105.859 | zero-shot=0.6803 | best=0.6803
[EP 10] loss=105.300 | zero-shot=0.6740 | best=0.6803

🎯 FEW-SHOT
🔥 FEW-SHOT: 0.7732884883880615

FOLD 2/13 | TEST SUBJECT = subject_02

🚀 TRAINING STUDENT (TRUE 3CH)
[EP 1] loss=120.219 | zero-shot=0.3962 | best=0.3962
[EP 2] loss=109.490 | zero-shot=0.4196 | best=0.4196
[EP 3] loss=107.663 | zero-shot=0.4643 | best=0.4643
[EP 4] loss=106.594 | zero-shot=0.4728 | best=0.4728
[EP 5] loss=105.420 | zero-shot=0.5112 | best=0.5112
[EP 6] loss=104.247 |

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# =========================
# CONFIG
# =========================
SELECTED_CH_IDX = [6, 7, 4]   # change electrodes here
# =========================
# CONVERT SUBJECTS TO INT IDS
# =========================
unique_subjects = np.unique(subjects)
subj_to_id = {s: i for i, s in enumerate(unique_subjects)}

subjects_num = np.array([subj_to_id[s] for s in subjects])

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

overall_results = []

# =========================
# BANDPOWER (LOCAL ONLY)
# =========================
def compute_bandpower(x):
    fft = torch.fft.rfft(x, dim=-1)
    psd = (fft.abs() ** 2)

    bands = [(1,4), (4,8), (8,13), (13,30)]

    bp = []
    for low, high in bands:
        bp.append(psd[..., low:high].mean(dim=-1))

    return torch.cat(bp, dim=1)  # [B, C*bands]

# =========================
# DATASET FIX (IMPORTANT)
# =========================
class LocalEEGDataset(torch.utils.data.Dataset):
    def __init__(self, X_time, y, subj):
        self.X_time = X_time
        self.y = y
        self.subj = subj

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_time[idx],
            self.y[idx],
            torch.tensor(self.subj[idx])  # 🔥 FIX HERE
        )

# =========================
# FOLD LOOP
# =========================
for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # SPLIT
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    # y_train = y_t[train_idx].clone()
    # y_test  = y_t[test_idx].clone()
    y_train = torch.randint_like(y_t[train_idx], low=0, high=3)
    y_test  = torch.randint_like(y_t[test_idx], low=0, high=3)

    # subj_train = subjects[train_idx]
    # subj_test  = subjects[test_idx]
    subj_train = subjects_num[train_idx]
    subj_test  = subjects_num[test_idx]

    # =========================
    # 🔥 SELECT CHANNELS FIRST
    # =========================
    X_time_train = X_time_train[:, SELECTED_CH_IDX, :]
    X_time_test  = X_time_test[:, SELECTED_CH_IDX, :]

    # =========================
    # NORMALIZATION
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6

    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    # DATASETS
    train_ds = LocalEEGDataset(X_time_train, y_train, subj_train)
    test_ds  = LocalEEGDataset(X_time_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)
    train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    student = StudentModel(emb_dim=64, proj_dim=64, bp_dim=3*4).to(device)
    opt = torch.optim.Adam(student.parameters(), lr=LR)

    best_state = None
    best_acc = -1

    print("\n🚀 TRAINING STUDENT (TRUE 3CH)")

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        student.train()
        total_loss = 0

        for xb_time, yb, sb in train_loader:

            xb_time = xb_time.to(device)
            yb = yb.to(device)
            sb = sb.to(device)  # ✅ NOW WORKS

            xb_bp = compute_bandpower(xb_time)

            e, proj = student(xb_time, xb_bp)

            loss_m = prototype_ce_loss(e, yb)
            loss_c = supcon_diff_subject_loss(proj, yb, sb)
            loss_p = prototype_loss(e, yb)

            loss = loss_m + LAMBDA*loss_c + PROTO_LAMBDA*loss_p

            opt.zero_grad()
            loss.backward()
            opt.step()

            total_loss += loss.item()

        # =========================
        # ZERO-SHOT
        # =========================
        student.eval()
        with torch.no_grad():

            def get_emb(loader):
                all_e, all_y = [], []
                for xb_time, yb, _ in loader:
                    xb_time = xb_time.to(device)
                    xb_bp = compute_bandpower(xb_time)
                    e, _ = student(xb_time, xb_bp)
                    all_e.append(e)
                    all_y.append(yb)
                return torch.cat(all_e), torch.cat(all_y)

            e_train, y_train_full = get_emb(train_eval_loader)
            e_test,  y_test_full  = get_emb(test_loader)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full==c].mean(0) for c in classes])

            logits = F.normalize(e_test,dim=1) @ F.normalize(protos,dim=1).T
            pred = logits.argmax(1)

            label_map = {int(c.item()):i for i,c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred.device
            )

            acc = (pred==y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(student.state_dict())

        print(f"[EP {ep}] loss={total_loss:.3f} | zero-shot={acc:.4f} | best={best_acc:.4f}")

    # =========================
    # LOAD BEST
    # =========================
    student.load_state_dict(best_state)

    # =========================
    # FEW-SHOT
    # =========================
    print("\n🎯 FEW-SHOT")

    student.eval()
    with torch.no_grad():
        e_all, y_all = get_emb(test_loader)

    support_idx = []
    for c in torch.unique(y_all):
        idx = (y_all == c).nonzero(as_tuple=True)[0]
        support_idx.append(idx[torch.randperm(len(idx))[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(y_all), dtype=torch.bool)
    mask[support_idx] = True

    e_support = F.normalize(e_all[mask].to(device), dim=1)
    e_query   = F.normalize(e_all[~mask].to(device), dim=1)

    y_support = y_all[mask].to(device)
    y_query   = y_all[~mask].to(device)

    protos = torch.stack([e_support[y_support==c].mean(0) for c in torch.unique(y_support)])
    protos = F.normalize(protos, dim=1)

    logits = e_query @ protos.T
    pred = logits.argmax(1)

    acc = (pred == y_query).float().mean().item()

    print("🔥 FEW-SHOT:", acc)

    overall_results.append(acc)

# =========================
# FINAL
# =========================
print("\nFINAL RESULTS")
print("Mean:", float(np.mean(overall_results)))
print("Std:", float(np.std(overall_results)))


FOLD 1/13 | TEST SUBJECT = subject_01

🚀 TRAINING STUDENT (TRUE 3CH)
[EP 1] loss=122.371 | zero-shot=0.3481 | best=0.3481
[EP 2] loss=111.697 | zero-shot=0.3333 | best=0.3481
[EP 3] loss=110.849 | zero-shot=0.3354 | best=0.3481
[EP 4] loss=110.600 | zero-shot=0.3291 | best=0.3481
[EP 5] loss=110.537 | zero-shot=0.3239 | best=0.3481
[EP 6] loss=110.474 | zero-shot=0.3260 | best=0.3481
[EP 7] loss=110.448 | zero-shot=0.3176 | best=0.3481
[EP 8] loss=110.437 | zero-shot=0.3270 | best=0.3481
[EP 9] loss=110.410 | zero-shot=0.3028 | best=0.3481
[EP 10] loss=110.405 | zero-shot=0.3228 | best=0.3481

🎯 FEW-SHOT
🔥 FEW-SHOT: 0.35465770959854126

FOLD 2/13 | TEST SUBJECT = subject_02

🚀 TRAINING STUDENT (TRUE 3CH)
[EP 1] loss=118.675 | zero-shot=0.3269 | best=0.3269
[EP 2] loss=111.539 | zero-shot=0.3280 | best=0.3280
[EP 3] loss=110.688 | zero-shot=0.3120 | best=0.3280
[EP 4] loss=110.464 | zero-shot=0.3184 | best=0.3280
[EP 5] loss=110.373 | zero-shot=0.3195 | best=0.3280
[EP 6] loss=110.337 

KeyboardInterrupt: 